In [ ]:
# import time
# import os
# import re
# import pandas as pd
# from dataclasses import dataclass, field, asdict
# from typing import Optional
# from openai import OpenAI

# # ─────────────────────────────────────────────────────────────
# # Wei et al. 2022 — Core Principles Implemented:
# #   1. Few-shot exemplars (8 per task) with hand-crafted chains
# #   2. Format: Q → reasoning steps → "The answer is X"
# #   3. Dataset-specific exemplar sets (arithmetic / commonsense)
# #   4. No model fine-tuning — pure prompting only
# #   5. Zero-shot fallback: "Let's think step by step"
# # ─────────────────────────────────────────────────────────────

# # ─────────────────────────────────────────────
# # 1. CONFIG
# # ─────────────────────────────────────────────
# LLM_CONFIG = {
#     "model":       "gpt-4o-mini",
#     "max_tokens":  2048,      # CoT needs more tokens for reasoning
#     "temperature": 0.0,       # deterministic — paper used greedy decoding
# }

# COST_PER_1K_TOKENS = {
#     "gpt-4o-mini":      {"input": 0.00015, "output": 0.0006},
#     "qwen-2-72b":       {"input": 0.0009,  "output": 0.0009},
#     "gemini-1.5-flash": {"input": 0.000075,"output": 0.0003},
#     "llama-3.1-70b":    {"input": 0.00059, "output": 0.00079},
# }

# PATHS = {
#     "gaia":      "../datasets/processed/processed_gaia.parquet",
#     "math_hard": "../datasets/processed/processed_math_hard.parquet",
#     "mmlu_pro":  "../datasets/processed/processed_mmlu_pro.parquet",
# }

# client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# # ─────────────────────────────────────────────
# # 2. DATA STRUCTURES
# # ─────────────────────────────────────────────
# @dataclass
# class Query:
#     id: str
#     dataset: str          # GAIA | MATH-HARD | MMLU-PRO
#     question: str
#     ground_truth: str
#     level: Optional[str]  = None
#     options: Optional[str]= None   # MMLU-Pro has MCQ options
#     subject: Optional[str]= None   # MMLU-Pro subject
#     metadata: dict        = field(default_factory=dict)

# @dataclass
# class Result:
#     query_id: str
#     dataset: str
#     question: str
#     ground_truth: str
#     full_reasoning: str   # complete CoT chain
#     predicted: str        # extracted final answer only
#     is_correct: bool
#     input_tokens: int
#     output_tokens: int
#     total_tokens: int
#     cost_usd: float
#     time_seconds: float
#     model: str
#     cot_type: str         # "few_shot" | "zero_shot"
#     level: Optional[str]  = None
#     subject: Optional[str]= None
#     error: Optional[str]  = None

# # ─────────────────────────────────────────────
# # 3. Wei et al. 2022 FEW-SHOT EXEMPLARS
# #    8 hand-crafted exemplars per dataset type
# #    Format strictly follows the paper:
# #    Q: ... \nA: <reasoning> The answer is <X>.
# # ─────────────────────────────────────────────

# # ── GAIA — real-world multi-step commonsense ──
# GAIA_EXEMPLARS = """Q: Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?
# A: Roger started with 5 balls. 2 cans × 3 balls = 6 balls. 5 + 6 = 11. The answer is 11.

# Q: A store sells apples at $1.50 each. A 10% discount is applied if you buy more than 4. How much do 6 apples cost?
# A: Cost without discount = 6 × $1.50 = $9.00. Discount = 10% of $9.00 = $0.90. Final cost = $9.00 − $0.90 = $8.10. The answer is 8.10.

# Q: A paper was submitted to arXiv in June 2022. It discusses AI regulation and includes a figure with axes labeled with opposing terms. What type of methodology involves structured oversight of automated decision systems?
# A: AI regulation papers typically discuss governance frameworks. Structured oversight of automated systems is referred to as algorithmic accountability. The answer is algorithmic accountability.

# Q: A train travels from City A to City B at 60 km/h and returns at 40 km/h. What is the average speed for the whole trip?
# A: Average speed for round trip = 2 × (s1 × s2) / (s1 + s2). = 2 × (60 × 40) / (60 + 40) = 4800 / 100 = 48 km/h. The answer is 48.

# Q: Which country won the most gold medals in the 2020 Summer Olympics?
# A: The 2020 Summer Olympics were held in Tokyo in 2021. The United States topped the medal table with 39 gold medals. The answer is United States.

# Q: A rectangle has a perimeter of 36 cm and a width of 8 cm. What is its area?
# A: Perimeter = 2 × (length + width). 36 = 2 × (length + 8). length + 8 = 18. length = 10. Area = 10 × 8 = 80 cm². The answer is 80.

# Q: If a species was introduced as a pet and later became invasive, what government agency in the US tracks its nonnative sightings?
# A: In the United States, the U.S. Geological Survey (USGS) maintains the Nonindigenous Aquatic Species (NAS) database which tracks nonnative species sightings. The answer is USGS.

# Q: What is the last letter of the last word in the phrase "chain of thought prompting"?
# A: The last word is "prompting". The last letter of "prompting" is "g". The answer is g."""


# # ── MATH-HARD — algebraic, geometry, number theory ──
# MATH_EXEMPLARS = """Q: What is the remainder when 2^100 is divided by 7?
# A: We find the pattern of 2^n mod 7. 2^1=2, 2^2=4, 2^3=8≡1 (mod 7). The cycle is [2,4,1] with period 3. 100 = 3×33 + 1, so 2^100 ≡ 2^1 = 2 (mod 7). The answer is 2.

# Q: Solve for x: 3x² − 12x + 9 = 0.
# A: Divide by 3: x² − 4x + 3 = 0. Factor: (x−1)(x−3) = 0. So x = 1 or x = 3. The answer is x = 1 or x = 3.

# Q: A right triangle has legs 5 and 12. What is the length of the hypotenuse?
# A: By Pythagorean theorem: h² = 5² + 12² = 25 + 144 = 169. h = √169 = 13. The answer is 13.

# Q: How many ways can 4 books be arranged on a shelf from 6 different books?
# A: This is a permutation: P(6,4) = 6!/(6−4)! = 6×5×4×3 = 360. The answer is 360.

# Q: What is the sum of the interior angles of a hexagon?
# A: Sum of interior angles = (n−2) × 180°. For hexagon n=6: (6−2) × 180 = 4 × 180 = 720°. The answer is 720.

# Q: Find the derivative of f(x) = 3x³ − 5x² + 2x − 7.
# A: Using power rule term by term: d/dx(3x³)=9x², d/dx(−5x²)=−10x, d/dx(2x)=2, d/dx(−7)=0. f'(x) = 9x² − 10x + 2. The answer is 9x² − 10x + 2.

# Q: If log₂(x) = 5, what is x?
# A: log₂(x) = 5 means 2⁵ = x. 2⁵ = 32. The answer is 32.

# Q: A circle has an area of 49π. What is its circumference?
# A: Area = πr² = 49π, so r² = 49, r = 7. Circumference = 2πr = 2π×7 = 14π. The answer is 14π."""


# # ── MMLU-PRO — multiple choice, domain knowledge ──
# MMLU_EXEMPLARS = """Q: Which of the following best describes the process of meiosis?
# Options: A) Cell growth and protein synthesis  B) DNA replication without cell division  C) Cell division that produces gametes with half the chromosome number  D) Mitotic division of somatic cells
# A: Meiosis is a specialized cell division that produces gametes (sperm and eggs). It results in four cells each with half the original chromosome number (haploid). This distinguishes it from mitosis which produces identical diploid cells. The answer is C.

# Q: What is the time complexity of binary search on a sorted array of n elements?
# Options: A) O(n)  B) O(log n)  C) O(n log n)  D) O(n²)
# A: Binary search works by repeatedly halving the search space. Starting with n elements, after 1 step: n/2, after 2 steps: n/4, after k steps: n/2^k = 1, so k = log₂(n). The answer is B.

# Q: In economics, what does the term "opportunity cost" refer to?
# Options: A) The direct monetary cost of a purchase  B) The value of the next best alternative foregone  C) The total cost of production  D) The marginal cost of one additional unit
# A: Opportunity cost is a core economic concept referring to what you give up by making a choice — specifically the value of the best alternative not chosen. It is not the direct monetary cost but the implicit cost of foregone options. The answer is B.

# Q: Which law states that the pressure of a gas is inversely proportional to its volume at constant temperature?
# Options: A) Charles's Law  B) Avogadro's Law  C) Boyle's Law  D) Gay-Lussac's Law
# A: Boyle's Law states P ∝ 1/V at constant temperature, i.e., P₁V₁ = P₂V₂. Charles's Law relates volume and temperature. Gay-Lussac's Law relates pressure and temperature. The answer is C.

# Q: Who wrote the philosophical work "Critique of Pure Reason"?
# Options: A) David Hume  B) René Descartes  C) John Locke  D) Immanuel Kant
# A: "Critique of Pure Reason" (Kritik der reinen Vernunft) was published in 1781 by Immanuel Kant. It is a foundational text in modern philosophy examining the nature and limits of human knowledge. The answer is D.

# Q: In Python, what does the "yield" keyword do?
# Options: A) Terminates a function immediately  B) Returns a value and pauses the function, making it a generator  C) Imports an external module  D) Declares a global variable
# A: "yield" turns a function into a generator. When called, it returns a value to the caller and suspends the function's state, allowing it to resume from that point on the next call. The answer is B.

# Q: What is the powerhouse of the cell?
# Options: A) Nucleus  B) Ribosome  C) Mitochondria  D) Golgi apparatus
# A: Mitochondria are responsible for producing ATP through cellular respiration, providing energy for cell functions. The nucleus stores DNA, ribosomes synthesize proteins, Golgi processes and packages proteins. The answer is C.

# Q: Which of the following is NOT a property of a normal distribution?
# Options: A) It is symmetric about the mean  B) Mean equals median equals mode  C) It has heavy tails compared to a uniform distribution  D) About 68% of data falls within 1 standard deviation of the mean
# A: A normal distribution is symmetric, has mean=median=mode, and follows the 68-95-99.7 rule. Normal distributions actually have lighter tails than heavy-tailed distributions like Cauchy. Option C incorrectly claims it has heavy tails — this is NOT a property. The answer is C."""


# # Map dataset → exemplar block
# EXEMPLARS = {
#     "GAIA":     GAIA_EXEMPLARS,
#     "MATH-HARD": MATH_EXEMPLARS,
#     "MMLU-PRO": MMLU_EXEMPLARS,
# }

# # ─────────────────────────────────────────────
# # 4. BUILD PROMPTS (Wei et al. style)
# # ─────────────────────────────────────────────
# def build_few_shot_prompt(query: Query) -> list[dict]:
#     """
#     Wei et al. 2022 Few-Shot CoT:
#     System sets the task context.
#     User message = 8 exemplars + new question.
#     """
#     exemplars = EXEMPLARS.get(query.dataset, GAIA_EXEMPLARS)

#     system = (
#         "You are an expert reasoning system. "
#         "For each question, think through it step by step, "
#         "then end with 'The answer is <answer>.'"
#     )

#     # For MMLU-Pro, append options to the question
#     question_text = query.question
#     if query.dataset == "MMLU-PRO" and query.options:
#         question_text = f"{query.question}\nOptions: {query.options}"

#     user = f"{exemplars}\n\nQ: {question_text}\nA:"

#     return [
#         {"role": "system", "content": system},
#         {"role": "user",   "content": user},
#     ]


# def build_zero_shot_prompt(query: Query) -> list[dict]:
#     """
#     Kojima et al. 2022 Zero-Shot CoT:
#     Append 'Let's think step by step' — used as fallback.
#     """
#     question_text = query.question
#     if query.dataset == "MMLU-PRO" and query.options:
#         question_text = f"{query.question}\nOptions: {query.options}"

#     return [
#         {"role": "system", "content": "You are an expert reasoning system."},
#         {"role": "user",   "content": f"Q: {question_text}\nA: Let's think step by step."},
#     ]

# # ─────────────────────────────────────────────
# # 5. ANSWER EXTRACTION
# #    Wei et al. always end with "The answer is X"
# # ─────────────────────────────────────────────
# def extract_answer(response_text: str, dataset: str) -> tuple[str, str]:
#     """
#     Returns (full_reasoning, extracted_answer)
#     Extracts from "The answer is X" pattern — Wei et al. 2022 standard.
#     """
#     full_reasoning = response_text.strip()

#     # Primary: "The answer is X." pattern (Wei et al. standard)
#     match = re.search(
#         r"[Tt]he answer is[:\s]+([^\n.]+)",
#         response_text
#     )
#     if match:
#         answer = match.group(1).strip().rstrip(".")
#         return full_reasoning, answer

#     # MMLU-Pro fallback: look for single letter A/B/C/D/E at end
#     if dataset == "MMLU-PRO":
#         match = re.search(r"\b([A-E])\b\.?\s*$", response_text.strip())
#         if match:
#             return full_reasoning, match.group(1)

#     # Last line fallback
#     lines  = [l.strip() for l in response_text.strip().split("\n") if l.strip()]
#     answer = lines[-1].rstrip(".") if lines else response_text
#     return full_reasoning, answer


# def normalize(text: str) -> str:
#     return text.strip().lower().rstrip(".,:;!")

# # ─────────────────────────────────────────────
# # 6. LOAD DATASETS
# # ─────────────────────────────────────────────
# def load_gaia(path: str) -> list[Query]:
#     df = pd.read_parquet(path)
#     print(f"[GAIA]      Loaded {len(df)} rows | Columns: {df.columns.tolist()}")
#     return [
#         Query(
#             id           = str(row.get("id", i)),
#             dataset      = "GAIA",
#             question     = str(row["query"]),
#             ground_truth = str(row["answer"]),
#             level        = str(row.get("level", "")),
#             metadata     = {"file": row.get("file_name"), "steps": row.get("steps_num")}
#         )
#         for i, (_, row) in enumerate(df.iterrows())
#     ]


# # def load_math_hard(path: str) -> list[Query]:
# #     df = pd.read_parquet(path)
# #     print(f"[MATH-HARD] Loaded {len(df)} rows | Columns: {df.columns.tolist()}")
# #     return [
# #         Query(
# #             id           = str(row.get("id", i)),
# #             dataset      = "MATH-HARD",
# #             question     = str(row.get("problem", row.get("question", ""))),
# #             ground_truth = str(row.get("answer",  row.get("solution", ""))),
# #             level        = str(row.get("level", "")),
# #             metadata     = {"type": row.get("type", "")}
# #         )
# #         for i, (_, row) in enumerate(df.iterrows())
# #     ]


# # def load_mmlu_pro(path: str) -> list[Query]:
# #     df = pd.read_parquet(path)
# #     print(f"[MMLU-PRO]  Loaded {len(df)} rows | Columns: {df.columns.tolist()}")
# #     queries = []
# #     for i, (_, row) in enumerate(df.iterrows()):
# #         # Build options string from columns (adjust col names to your parquet)
# #         options = row.get("options", None)
# #         if isinstance(options, list):
# #             labels  = ["A", "B", "C", "D", "E", "F", "G", "H"]
# #             options = "  ".join(f"{labels[j]}) {opt}" for j, opt in enumerate(options))

# #         queries.append(Query(
# #             id           = str(row.get("question_id", i)),
# #             dataset      = "MMLU-PRO",
# #             question     = str(row.get("question", "")),
# #             ground_truth = str(row.get("answer", row.get("answer_index", ""))),
# #             level        = str(row.get("category", row.get("subject", ""))),
# #             options      = options,
# #             subject      = str(row.get("category", row.get("subject", ""))),
# #         ))
# #     return queries


# def load_all() -> list[Query]:
#     queries = []
#     queries += load_gaia(PATHS["gaia"])
#     # queries += load_math_hard(PATHS["math_hard"])
#     # queries += load_mmlu_pro(PATHS["mmlu_pro"])
#     print(f"\nTotal: {len(queries)} queries across 3 datasets\n")
#     return queries

# # ─────────────────────────────────────────────
# # 7. CoT LLM CALL — Wei et al. 2022
# # ─────────────────────────────────────────────
# def call_cot_llm(query: Query, cot_type: str = "few_shot") -> Result:
#     model   = LLM_CONFIG["model"]
#     start   = time.perf_counter()
#     error   = None
#     full_reasoning = ""
#     predicted      = ""
#     usage   = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

#     messages = (
#         build_few_shot_prompt(query)
#         if cot_type == "few_shot"
#         else build_zero_shot_prompt(query)
#     )

#     try:
#         response = client.chat.completions.create(
#             model       = model,
#             max_tokens  = LLM_CONFIG["max_tokens"],
#             temperature = LLM_CONFIG["temperature"],   # greedy — Wei et al. 2022
#             messages    = messages,
#         )
#         raw_text = response.choices[0].message.content.strip()
#         full_reasoning, predicted = extract_answer(raw_text, query.dataset)

#         usage = {
#             "prompt_tokens":     response.usage.prompt_tokens,
#             "completion_tokens": response.usage.completion_tokens,
#             "total_tokens":      response.usage.total_tokens,
#         }
#     except Exception as e:
#         error = str(e)

#     elapsed  = time.perf_counter() - start
#     rates    = COST_PER_1K_TOKENS.get(model, {"input": 0, "output": 0})
#     cost     = (
#         usage["prompt_tokens"]     / 1000 * rates["input"] +
#         usage["completion_tokens"] / 1000 * rates["output"]
#     )
#     is_correct = normalize(predicted) == normalize(query.ground_truth)

#     return Result(
#         query_id       = query.id,
#         dataset        = query.dataset,
#         question       = query.question,
#         ground_truth   = query.ground_truth,
#         full_reasoning = full_reasoning,
#         predicted      = predicted,
#         is_correct     = is_correct,
#         input_tokens   = usage["prompt_tokens"],
#         output_tokens  = usage["completion_tokens"],
#         total_tokens   = usage["total_tokens"],
#         cost_usd       = round(cost, 6),
#         time_seconds   = round(elapsed, 3),
#         model          = model,
#         cot_type       = cot_type,
#         level          = query.level,
#         subject        = query.subject,
#         error          = error,
#     )

# # ─────────────────────────────────────────────
# # 8. RUN EXPERIMENT
# # ─────────────────────────────────────────────
# def run_cot_experiment(
#     queries:  list[Query],
#     cot_type: str = "few_shot"   # "few_shot" | "zero_shot"
# ) -> list[Result]:

#     print(f"\n{'='*60}")
#     print(f"  Wei et al. 2022 CoT — {cot_type.upper()}")
#     print(f"  Model  : {LLM_CONFIG['model']}")
#     print(f"  Queries: {len(queries)}")
#     print(f"{'='*60}\n")

#     results = []
#     for i, q in enumerate(queries):
#         print(f"[{i+1}/{len(queries)}] {q.dataset} | Level: {q.level} | {q.id}")
#         print(f"   ❓ Q  : {q.question[:100]}...")
#         print(f"   ✅ GT : {q.ground_truth}")

#         r = call_cot_llm(q, cot_type=cot_type)
#         results.append(r)

#         status = "✓" if r.is_correct else "✗"
#         print(f"   🧠 CoT: {r.full_reasoning[:120].strip()}...")
#         print(f"   🤖 Ans: {r.predicted}")
#         print(f"   {status}  tokens={r.total_tokens}  cost=${r.cost_usd:.5f}  "
#               f"time={r.time_seconds}s")
#         if r.error:
#             print(f"   ⚠ {r.error}")
#         print("-" * 65)

#     return results

# # ─────────────────────────────────────────────
# # 9. EVALUATE — level + subject breakdown
# # ─────────────────────────────────────────────
# def evaluate(results: list[Result]):
#     df = pd.DataFrame([asdict(r) for r in results])

#     print(f"\n{'='*60}")
#     print(f"  RESULTS — Wei et al. 2022 CoT ({results[0].cot_type})")
#     print(f"  Model: {results[0].model}")
#     print(f"{'='*60}")
#     print(f"  Total    : {len(df)}")
#     print(f"  Accuracy : {df['is_correct'].sum()}/{len(df)} = {df['is_correct'].mean():.1%}")
#     print(f"  Tokens   : {df['total_tokens'].sum():,}")
#     print(f"  Cost     : ${df['cost_usd'].sum():.4f}")
#     print(f"  Latency  : {df['time_seconds'].mean():.2f}s avg")
#     print(f"  Errors   : {df['error'].notna().sum()}")

#     for dataset in sorted(df["dataset"].unique()):
#         ddf = df[df["dataset"] == dataset]
#         print(f"\n─── {dataset} (n={len(ddf)}) ─────────────────────────")

#         group_col = "subject" if dataset == "MMLU-PRO" else "level"
#         if group_col in ddf.columns and ddf[group_col].notna().any():
#             grp = (
#                 ddf.groupby(group_col)
#                 .agg(correct=("is_correct","sum"),
#                      total  =("is_correct","count"),
#                      acc    =("is_correct","mean"),
#                      tokens =("total_tokens","mean"))
#                 .sort_values("acc", ascending=False)
#                 .reset_index()
#             )
#             print(f"  {group_col:<20} {'Correct':<9} {'Total':<9} {'Acc':<9} {'AvgTok'}")
#             print(f"  {'-'*19} {'-'*8} {'-'*8} {'-'*8} {'-'*7}")
#             for _, row in grp.iterrows():
#                 print(f"  {str(row[group_col]):<20} "
#                       f"{int(row['correct']):<9} "
#                       f"{int(row['total']):<9} "
#                       f"{row['acc']:<9.1%} "
#                       f"{row['tokens']:.0f}")

#         # Dataset total
#         print(f"\n  TOTAL  acc={ddf['is_correct'].mean():.1%}  "
#               f"cost=${ddf['cost_usd'].sum():.4f}")

#     return df

# # ─────────────────────────────────────────────
# # 10. SAVE
# # ─────────────────────────────────────────────
# def save_results(results: list[Result], dataset_name: str, cot_type: str):
#     model_tag = LLM_CONFIG["model"].replace("/", "-")
#     save_dir  = os.path.join("..", "datasets", "baseline", dataset_name.lower())
#     os.makedirs(save_dir, exist_ok=True)

#     # e.g. v1_wei2022_fewshot_gpt-4o-mini.parquet
#     fname        = f"v1_wei2022_{cot_type}_{model_tag}.parquet"
#     parquet_path = os.path.join(save_dir, fname)

#     pd.DataFrame([asdict(r) for r in results]).to_parquet(parquet_path, index=False)
#     print(f"\n✅ Saved → {parquet_path}  ({len(results)} rows)")

# # ─────────────────────────────────────────────
# # 11. MAIN
# # ─────────────────────────────────────────────
# if __name__ == "__main__":
#     all_queries = load_all()

#     COT_TYPE = "few_shot"    # swap to "zero_shot" for Kojima et al. style

#     for dataset_name in ["GAIA", "MATH-HARD", "MMLU-PRO"]:
#         queries = [q for q in all_queries if q.dataset == dataset_name]
#         if not queries:
#             print(f"[SKIP] No queries found for {dataset_name}")
#             continue

#         results = run_cot_experiment(queries, cot_type=COT_TYPE)
#         evaluate(results)
#         save_results(results, dataset_name, COT_TYPE)
